# A Predictive Model for US Airline Flight Delays - Basedline

### <span style="color:chocolate"> Project Description </span>

Flight delay prediction is an important research field, as flight delays are the primary concern of aviation
stakeholders. Our project aims to identify the predictors of flight delays before they happen, improve
travelers experience, and help airlines shift from reactive damage control to proactive delay management.

---
### <span style="color:chocolate">Import libraries</span>

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

/Users/rosalinlun/Documents/DATASCI_207_Applied_Machine_Learning/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


---
### <span style="color:chocolate">Data Ingestion</span>

Reading the cleaned and combined data from 2022 to 2026 to the data frame downloaded from the following data sources:
- Flight data: Bureau of transportation stats https://www.transtats.bts.gov/ontime/
- Weather data (New with 2026 data): https://www.ncei.noaa.gov/access/search/data-search/global-historical-climatology-network-hourly
- Aircraft data: https://www.faa.gov/licenses_certificates/aircraft_certification/aircraft_registry/releasable_aircraft_download?utm_source=chatgpt.com

In [3]:
# Read in the data.
flight_df = pd.read_csv('Data/combined_data_1h.csv')

In [4]:
# Display the first five rows of data.
flight_df.head()

,date,scheduled_dep_dt,actual_dep_dt,carrier_code,destination_airport,scheduled_elapsed_time_minutes,year,month,day_of_week,is_weekend,...,dew_point_temperature,relative_humidity,altimeter,aircraft_age,week_num,temperature_dewpoint_spread,airport_delay_average_1h,airport_delay_stddev_1h,airport_departures_observed_1h,departure_delay_minutes
0,2022-01-01,2022-01-01 05:25:00,2022-01-01 05:25:00,WN,DEN,145.0,2022.0,1.0,5.0,1,...,5.0,74.0,1014.6,16.0,1,4.4,0.0,0.0,0.0,0.0
1,2022-01-01,2022-01-01 05:55:00,2022-01-01 05:52:00,DL,SLC,118.0,2022.0,1.0,5.0,1,...,5.0,74.0,1014.6,3.0,1,4.4,0.0,0.0,0.0,-3.0
2,2022-01-01,2022-01-01 06:00:00,2022-01-01 06:14:00,DL,ATL,272.0,2022.0,1.0,5.0,1,...,5.0,77.0,1015.6,20.0,1,3.9,0.0,0.0,0.0,14.0
3,2022-01-01,2022-01-01 06:02:00,2022-01-01 05:56:00,UA,LAX,98.0,2022.0,1.0,5.0,1,...,5.0,77.0,1015.6,6.0,1,3.9,0.0,0.0,0.0,-6.0
4,2022-01-01,2022-01-01 06:10:00,2022-01-01 07:51:00,DL,MSP,218.0,2022.0,1.0,5.0,1,...,5.0,77.0,1015.6,24.0,1,3.9,0.0,0.0,0.0,101.0


In [5]:
# Display the shape
flight_df.shape

(334086, 32)

In [6]:
# Display the max departure delay minutes
flight_df['departure_delay_minutes'].max()

np.float64(1101.0)

In [7]:
# Check the types of each column
flight_df.dtypes

date                               object
scheduled_dep_dt                   object
actual_dep_dt                      object
carrier_code                       object
destination_airport                object
scheduled_elapsed_time_minutes    float64
year                              float64
month                             float64
day_of_week                       float64
is_weekend                          int64
sched_dep_hour                    float64
outcome                           float64
days_until_holiday                float64
days_from_holiday                 float64
year_mfr                          float64
model                              object
no_seats                          float64
visibility                        float64
ceiling_height                    float64
wind_speed                        float64
wind_direction                    float64
temperature                       float64
dew_point_temperature             float64
relative_humidity                 

In [8]:
# Check if there is any early arrival flight
(flight_df['departure_delay_minutes'] < 0).any().any()
print(flight_df.loc[flight_df['departure_delay_minutes'] < 0, 'departure_delay_minutes'].tolist())

[-3.0, -6.0, -6.0, -6.0, -4.0, -2.0, -6.0, -5.0, -4.0, -1.0, -8.0, -4.0, -7.0, -2.0, -3.0, -6.0, -7.0, -2.0, -3.0, -4.0, -4.0, -3.0, -4.0, -9.0, -3.0, -8.0, -1.0, -4.0, -11.0, -9.0, -3.0, -5.0, -8.0, -5.0, -3.0, -5.0, -2.0, -2.0, -2.0, -5.0, -13.0, -4.0, -3.0, -3.0, -2.0, -1.0, -3.0, -4.0, -7.0, -1.0, -2.0, -1.0, -5.0, -2.0, -2.0, -4.0, -2.0, -2.0, -6.0, -6.0, -5.0, -3.0, -2.0, -3.0, -8.0, -8.0, -10.0, -2.0, -4.0, -1.0, -3.0, -6.0, -4.0, -5.0, -5.0, -5.0, -3.0, -5.0, -4.0, -6.0, -8.0, -5.0, -3.0, -1.0, -4.0, -3.0, -5.0, -5.0, -8.0, -1.0, -4.0, -8.0, -1.0, -3.0, -2.0, -6.0, -3.0, -4.0, -2.0, -2.0, -1.0, -3.0, -6.0, -5.0, -2.0, -18.0, -2.0, -9.0, -4.0, -5.0, -2.0, -3.0, -6.0, -3.0, -2.0, -1.0, -2.0, -3.0, -1.0, -9.0, -4.0, -2.0, -4.0, -8.0, -4.0, -2.0, -5.0, -2.0, -3.0, -1.0, -4.0, -2.0, -6.0, -3.0, -5.0, -2.0, -2.0, -8.0, -7.0, -11.0, -3.0, -7.0, -5.0, -5.0, -3.0, -3.0, -6.0, -2.0, -10.0, -5.0, -1.0, -3.0, -3.0, -6.0, -7.0, -8.0, -4.0, -7.0, -2.0, -1.0, -7.0, -4.0, -8.0, -9.0, -2.0, -6.

---
#### <span style="color:chocolate">Data Preprocessing</span>



In [9]:
# Assigned the duration of departure delay into different category.
def delay_category(row):
    delay = row['departure_delay_minutes']
    # (1) No delay
    if delay <= 0:
        return 0
    # (2) Less than 1 hour
    elif delay < 60:
        return 1
    # (3) 2 - 3 hour
    elif delay < 180:
        return 2
    # (4) 3 - 4 hour
    elif delay < 240:
        return 3
    # (5) 4 - 5 hours
    elif delay < 300:
        return 4
    # (6) 5 - 6 hours
    elif delay < 360:
        return 5
    # (7) More than 6 hours
    elif delay > 360:
        return 6
    # (8) Cancelled
    else:
        return 7 # >6 hours

In [10]:
# Create a new column in the dataframe to save the target output.
flight_df['target'] = flight_df.apply(delay_category, axis=1)

In [11]:
# Display the first five rows of data.
flight_df.head()

,date,scheduled_dep_dt,actual_dep_dt,carrier_code,destination_airport,scheduled_elapsed_time_minutes,year,month,day_of_week,is_weekend,...,relative_humidity,altimeter,aircraft_age,week_num,temperature_dewpoint_spread,airport_delay_average_1h,airport_delay_stddev_1h,airport_departures_observed_1h,departure_delay_minutes,target
0,2022-01-01,2022-01-01 05:25:00,2022-01-01 05:25:00,WN,DEN,145.0,2022.0,1.0,5.0,1,...,74.0,1014.6,16.0,1,4.4,0.0,0.0,0.0,0.0,0
1,2022-01-01,2022-01-01 05:55:00,2022-01-01 05:52:00,DL,SLC,118.0,2022.0,1.0,5.0,1,...,74.0,1014.6,3.0,1,4.4,0.0,0.0,0.0,-3.0,0
2,2022-01-01,2022-01-01 06:00:00,2022-01-01 06:14:00,DL,ATL,272.0,2022.0,1.0,5.0,1,...,77.0,1015.6,20.0,1,3.9,0.0,0.0,0.0,14.0,1
3,2022-01-01,2022-01-01 06:02:00,2022-01-01 05:56:00,UA,LAX,98.0,2022.0,1.0,5.0,1,...,77.0,1015.6,6.0,1,3.9,0.0,0.0,0.0,-6.0,0
4,2022-01-01,2022-01-01 06:10:00,2022-01-01 07:51:00,DL,MSP,218.0,2022.0,1.0,5.0,1,...,77.0,1015.6,24.0,1,3.9,0.0,0.0,0.0,101.0,2


In [12]:
# Group the numeric features together.
features_numeric = [
    'scheduled_elapsed_time_minutes', 'year', 'month', 'day_of_week', 'is_weekend',
    'sched_dep_hour', 'days_until_holiday', 'days_from_holiday', 'year_mfr', 'no_seats',
    'visibility', 'ceiling_height', 'wind_speed', 'wind_direction', 'temperature',
    'dew_point_temperature', 'relative_humidity', 'altimeter', 'aircraft_age', 'week_num',
    'temperature_dewpoint_spread', 'airport_delay_average_1h', 'airport_delay_stddev_1h',
    'airport_departures_observed_1h'
]

In [13]:
# Group the categorical features together.
features_categorical = ['carrier_code', 'destination_airport', 'model']

In [14]:
# Group the datetime feature.
datetime_feature = ['date']

In [15]:
# Divide the data into input and output.
X = flight_df[datetime_feature + features_numeric + features_categorical].copy()
y = flight_df['target'].values

In [16]:
# Display the first five rows of X.
X.head()

,date,scheduled_elapsed_time_minutes,year,month,day_of_week,is_weekend,sched_dep_hour,days_until_holiday,days_from_holiday,year_mfr,...,altimeter,aircraft_age,week_num,temperature_dewpoint_spread,airport_delay_average_1h,airport_delay_stddev_1h,airport_departures_observed_1h,carrier_code,destination_airport,model
0,2022-01-01,145.0,2022.0,1.0,5.0,1,5.0,0.0,0.0,2006.0,...,1014.6,16.0,1,4.4,0.0,0.0,0.0,WN,DEN,737-76N
1,2022-01-01,118.0,2022.0,1.0,5.0,1,5.0,0.0,0.0,2019.0,...,1014.6,3.0,1,4.4,0.0,0.0,0.0,DL,SLC,BD-500-1A10
2,2022-01-01,272.0,2022.0,1.0,5.0,1,6.0,0.0,0.0,2002.0,...,1015.6,20.0,1,3.9,0.0,0.0,0.0,DL,ATL,757-351
3,2022-01-01,98.0,2022.0,1.0,5.0,1,6.0,0.0,0.0,2016.0,...,1015.6,6.0,1,3.9,0.0,0.0,0.0,UA,LAX,737-824
4,2022-01-01,218.0,2022.0,1.0,5.0,1,6.0,0.0,0.0,1998.0,...,1015.6,24.0,1,3.9,0.0,0.0,0.0,DL,MSP,A320-212


In [17]:
# Sort the row by the values in 'date' column
flight_df = flight_df.sort_values('date').reset_index(drop=True)

In [18]:
# Calculate the number of samples based on the split percentage then save the indexes number.
total_rows = len(flight_df)
train_end_idx = int(total_rows * 0.60)
test_end_idx = train_end_idx + int(total_rows * 0.20)

# Split the data
X_train = X[:train_end_idx].copy()
y_train = y[:train_end_idx].copy()
X_test = X[train_end_idx:test_end_idx].copy()
y_test = y[train_end_idx:test_end_idx].copy()
X_val = X[test_end_idx:].copy()
y_val = y[test_end_idx:].copy()

In [19]:
# Display the first five rows of X_train.
X_train.head()

,date,scheduled_elapsed_time_minutes,year,month,day_of_week,is_weekend,sched_dep_hour,days_until_holiday,days_from_holiday,year_mfr,...,altimeter,aircraft_age,week_num,temperature_dewpoint_spread,airport_delay_average_1h,airport_delay_stddev_1h,airport_departures_observed_1h,carrier_code,destination_airport,model
0,2022-01-01,145.0,2022.0,1.0,5.0,1,5.0,0.0,0.0,2006.0,...,1014.6,16.0,1,4.4,0.0,0.0,0.0,WN,DEN,737-76N
1,2022-01-01,118.0,2022.0,1.0,5.0,1,5.0,0.0,0.0,2019.0,...,1014.6,3.0,1,4.4,0.0,0.0,0.0,DL,SLC,BD-500-1A10
2,2022-01-01,272.0,2022.0,1.0,5.0,1,6.0,0.0,0.0,2002.0,...,1015.6,20.0,1,3.9,0.0,0.0,0.0,DL,ATL,757-351
3,2022-01-01,98.0,2022.0,1.0,5.0,1,6.0,0.0,0.0,2016.0,...,1015.6,6.0,1,3.9,0.0,0.0,0.0,UA,LAX,737-824
4,2022-01-01,218.0,2022.0,1.0,5.0,1,6.0,0.0,0.0,1998.0,...,1015.6,24.0,1,3.9,0.0,0.0,0.0,DL,MSP,A320-212


In [20]:
# Verify the date range is splitted correctly
print(f"Train: {X_train['date'].min()} to {X_train['date'].max()}")
print(f"Test: {X_test['date'].min()} to {X_test['date'].max()}")
print(f"Val: {X_val['date'].min()} to {X_val['date'].max()}")

Train: 2022-01-01 to 2024-08-07
Test: 2024-08-07 to 2025-05-16
Val: 2025-05-16 to 2026-04-30


In [21]:
# Use sklearn StandarScaler to standardize the numeric features, such that they all have a mean of 0 and a standard deviation of 1.
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[features_numeric])
X_val_num = scaler.transform(X_val[features_numeric])
X_test_num = scaler.transform(X_test[features_numeric])

In [22]:
# Use sklearn OrdinalEncoder to convert categorical data to numeric features.
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_cat = encoder.fit_transform(X_train[features_categorical]) + 1
X_val_cat = encoder.transform(X_val[features_categorical]) + 1
X_test_cat = encoder.transform(X_test[features_categorical]) + 1

In [23]:
# Helper function to format the processed categorical features/data into a dictionary of NumPy arrays for training the model with Keras.
"""
Params:
-------
num_arr: Array of standardized numerical features
cat_arr: Array of Converted categorical features
cat_cols: Categorical column names

Returns:
--------
input_dict: Dictionary of NumPy arrays
"""
def categorical_dict(num_arr, cat_arr, cat_cols):
    input_dict = {'numeric_inputs': num_arr}
    for i, col in enumerate(cat_cols):
        input_dict[col] = cat_arr[:, i]
    return input_dict

In [24]:
train_inputs = categorical_dict(X_train_num, X_train_cat, features_categorical)
val_inputs = categorical_dict(X_val_num, X_val_cat, features_categorical)
test_inputs = categorical_dict(X_test_num, X_test_cat, features_categorical)

---
### <span style="color:chocolate">Baseline Model</span>

In [25]:
# Initial the number of perdict output categories for the baseline multinomial logistic regression model.
num_classes = 8
# Create an empty list to store the Keras input layers
input_layers = []
# Create an empty list to store the transformed outputs
processed_features = []

In [26]:
# Create the input layer for the numerical features.
num_input = tf.keras.Input(shape=(len(features_numeric),), name='numeric_inputs')
input_layers.append(num_input)
processed_features.append(num_input)

In [27]:
# Loop through all the categorical features and process them by using the CategoryEncoding() in Keras.
for col in features_categorical:
    vocab_size = int(max(X_train_cat[:, features_categorical.index(col)]) + 1)
    cat_input = tf.keras.Input(shape=(1,), name=col, dtype=tf.int32)
    input_layers.append(cat_input)

    # Convert tokens to one-hot encoding
    one_hot = tf.keras.layers.CategoryEncoding(num_tokens=vocab_size, output_mode="one_hot")(cat_input)
    # Squeeze out the extra dimension caused by the sequence length of 1
    one_hot = tf.keras.layers.Reshape((vocab_size,))(one_hot)
    processed_features.append(one_hot)

In [28]:
# Combine the numerical and categorical features into a single vector
concat_features = tf.keras.layers.concatenate(processed_features)

In [29]:
# The output/final layer of the model
outputs = tf.keras.layers.Dense(
    units=num_classes,
    activation='softmax',
    # Add L2 regularization to avoid overfitting due to imbalance class
    kernel_regularizer=tf.keras.regularizers.l2(0.001),
    name='output'
)(concat_features)

model = tf.keras.Model(inputs=input_layers, outputs=outputs)

In [30]:
# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    # Used because targets are integers (0-8)
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

print(model.summary())

# Train the model
history = model.fit(
    x=train_inputs,
    y=y_train,
    validation_data=(val_inputs, y_val),
    batch_size=512,
    epochs=10
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ carrier_code        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ destination_airport │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ model (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ category_encoding   │ (None, 9)         │          0 │ carrier_code[0][… │
│ (CategoryEncoding)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ category_encoding_1 │ (None, 69)        │          0 │ destination_airp… │
│ (CategoryEncoding)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ category_encoding_2 │ (None, 78)        │          0 │ model[0][0]       │
│ (CategoryEncoding)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_inputs      │ (None, 24)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 9)         │          0 │ category_encodin… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 69)        │          0 │ category_encodin… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, 78)        │          0 │ category_encodin… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 180)       │          0 │ numeric_inputs[0… │
│ (Concatenate)       │                   │            │ reshape[0][0],    │
│                     │                   │            │ reshape_1[0][0],  │
│                     │                   │            │ reshape_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 8)         │      1,448 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,448 (5.66 KB)

 Trainable params: 1,448 (5.66 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 1s 843us/step - accuracy: 0.4150 - loss: 1.7126 - val_accuracy: 0.5921 - val_loss: 1.1311
Epoch 2/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 561us/step - accuracy: 0.6074 - loss: 1.0562 - val_accuracy: 0.6263 - val_loss: 0.9997
Epoch 3/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 614us/step - accuracy: 0.6182 - loss: 0.9636 - val_accuracy: 0.6291 - val_loss: 0.9591
Epoch 4/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 559us/step - accuracy: 0.6222 - loss: 0.9345 - val_accuracy: 0.6308 - val_loss: 0.9376
Epoch 5/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 573us/step - accuracy: 0.6230 - loss: 0.9160 - val_accuracy: 0.6329 - val_loss: 0.9250
Epoch 6/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 549us/step - accuracy: 0.6219 - loss: 0.9034 - val_accuracy: 0.6329 - val_loss: 0.9142
Epoch 7/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 546us/step - accuracy: 0.6231 - loss: 0.8953 - val_accuracy: 0.6345 - val_loss: 0.9029
Epoch 8/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 568us/step - accuracy: 0.6226 - loss: 0.88